# Duration Vertical-Slice MVP — Stage 1

Run all cells from the repository root (`poetry run jupyter lab`). Requires the project's Python environment, PyYAML, and internet access for the first run. No FRED key or manually prepared inputs are needed.

**Raw Observations → Features → Component Values / States → Duration Bond Exposure View**

This provisional model implements only Stage 1 of [the MVP contract](docs/bondview_duration_vertical_slice_mvp.md). Each boundary is retained per historical date. No ETF Exposure Profiles or ETF Evaluation are implemented.

## Provisional economic assumptions

- **DGS10**: 10Y constant-maturity yield represents long-end conditions; DGS30 may describe ultra-long exposure differently.
- **CPIAUCSL**: seasonally adjusted headline CPI. Inflation Trend is the three-month change in year-over-year inflation, not the inflation level. Core CPI or PCE can change the interpretation.
- **DFEDTARU**: target range upper limit represents realized policy moves, not expectations. A 2Y yield or futures measure could recognize a transition before a target change.
- Long-End Yield Trend compares 5-observation yield averages 63 valid observations apart; Recent Long-End Yield Move uses 21 observations. Policy Direction is the 90-calendar-day change. CPI uses monthly observations.
- States use symmetric neutral bands (±25 bp trend, ±15 bp recent move, ±0.25 percentage points inflation, ±12.5 bp policy), including exact boundaries. Horizons and thresholds are uncalibrated YAML assumptions.
- View contributions are +1 for falling yields / cooling inflation / easing, -1 for their opposites, and 0 for neutral states. A tally ≥2 is `duration_supportive`, ≤-2 is `duration_headwind`, otherwise `mixed`. This covers all 81 cases. It is a transparent descriptive mapping, not a return forecast or ETF recommendation. The two overlapping yield signals give rates half the possible contribution and may double-count momentum.

## Historical availability

Download directly from FRED's [ALFRED vintage service](https://alfred.stlouisfed.org/help), selecting `vintage_date=as_of` for **every** input, including rates. This excludes observations and revisions absent from that vintage. Dates represent end-of-day archive availability, not an intraday trading timestamp. Archive publication may lag the source; no synthetic release lag or current revised CPI is substituted. The returned vintage header is checked. Source corrections to archived vintages remain possible.

References: [DGS10](https://fred.stlouisfed.org/series/DGS10), [CPIAUCSL](https://fred.stlouisfed.org/series/CPIAUCSL), [DFEDTARU](https://fred.stlouisfed.org/series/DFEDTARU).


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from hashlib import sha256
from io import BytesIO
import json
from urllib.parse import urlencode
from urllib.request import urlopen

import pandas as pd
from IPython.display import display
from duration_model import load_config, calculate_features, calculate_components, calculate_view

config = load_config("duration_mvp.yaml")
display(pd.DataFrame(config["series"]).T)
display(pd.DataFrame(config["components"]).T)
config["view"]

,id,frequency,units,source,max_age_days,assumption
long_yield,DGS10,business_daily,percent,"Board of Governors, H.15",7,10Y Treasury is a liquid long-end proxy; DGS30...
inflation,CPIAUCSL,monthly,index_1982_84_100,"Bureau of Labor Statistics, seasonally adjuste...",75,Headline CPI captures broad inflation; core CP...
policy,DFEDTARU,daily,percent,"Board of Governors, FOMC target range upper limit",7,Realized target changes proxy policy direction...


,feature,scale,units,threshold,states
Long-End Yield Trend,yield_trend_endpoints,100,bp,25,"[falling, stable, rising]"
Recent Long-End Yield Move,yield_move_endpoints,100,bp,15,"[falling, stable, rising]"
Inflation Trend,inflation_trend_endpoints,1,percentage_points,0.25,"[cooling, stable, heating]"
Policy Direction,policy_direction_endpoints,100,bp,12.5,"[easing, unchanged, tightening]"


{'contributions': {'Long-End Yield Trend': {'falling': 1,
   'stable': 0,
   'rising': -1},
  'Recent Long-End Yield Move': {'falling': 1, 'stable': 0, 'rising': -1},
  'Inflation Trend': {'cooling': 1, 'stable': 0, 'heating': -1},
  'Policy Direction': {'easing': 1, 'unchanged': 0, 'tightening': -1}},
 'threshold': 2,
 'labels': ['duration_headwind', 'mixed', 'duration_supportive']}

## Source retrieval and normalized Raw Observations

This notebook-local helper owns retrieval; calculation code has no network or file-cache dependency. Original CSV bytes plus URL, SHA-256, retrieval time, and vintage are cached under ignored `_private/duration_mvp/`. Delete that local cache or set `refresh=True` to retrieve again. Cache hits verify metadata and bytes; missing downloads fail explicitly without fabricated fallback data. The full normalized date-indexed series remain accessible through `raw_by_date[as_of][series_alias]`. Missing market holidays are retained in Raw Observations and omitted only when calculating valid-observation windows.

In [2]:
cache_dir = Path("_private/duration_mvp")
cache_dir.mkdir(parents=True, exist_ok=True)

def download_vintage(spec, as_of, refresh=False):
    params = {"id": spec["id"], "cosd": config["history_start"],
              "coed": as_of, "vintage_date": as_of}
    url = "https://alfred.stlouisfed.org/graph/alfredgraph.csv?" + urlencode(params)
    key = sha256(url.encode()).hexdigest()[:20]
    csv_path, meta_path = cache_dir / f"{key}.csv", cache_dir / f"{key}.json"
    if not refresh and csv_path.exists() and meta_path.exists():
        payload, metadata = csv_path.read_bytes(), json.loads(meta_path.read_text())
        if metadata["url"] != url or metadata["sha256"] != sha256(payload).hexdigest():
            raise ValueError(f"Invalid cached data: {csv_path}; refresh explicitly")
    else:
        try:
            with urlopen(url, timeout=45) as response:
                payload = response.read()
        except Exception as exc:
            raise RuntimeError(f"FRED vintage retrieval failed: {url}") from exc
        metadata = {"series_id": spec["id"], "vintage_date": as_of, "url": url,
                    "source": spec["source"], "units": spec["units"],
                    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
                    "sha256": sha256(payload).hexdigest()}
    frame = pd.read_csv(BytesIO(payload), na_values=["."])
    expected = f"{spec['id']}_{as_of.replace('-', '')}"
    if list(frame.columns) != ["observation_date", expected]:
        raise ValueError(f"Unexpected vintage response for {url}: {list(frame.columns)}")
    dates = pd.DatetimeIndex(pd.to_datetime(frame["observation_date"], errors="raise"))
    if dates.has_duplicates or dates.hasnans or not dates.is_monotonic_increasing:
        raise ValueError(f"Invalid observation dates: {url}")
    if dates.empty or dates.min() < pd.Timestamp(config["history_start"]) or dates.max() > pd.Timestamp(as_of):
        raise ValueError(f"Out-of-range observation dates: {url}")
    series = pd.Series(pd.to_numeric(frame[expected], errors="raise").to_numpy(),
                       index=dates, name=spec["id"])
    series.attrs.update(metadata)
    csv_path.write_bytes(payload)
    meta_path.write_text(json.dumps(metadata, indent=2) + "\n")
    return series

raw_by_date = {}
coverage_rows = []
for as_of in config["as_of_dates"]:
    raw_by_date[as_of] = {}
    for alias, spec in config["series"].items():
        series = download_vintage(spec, as_of)
        raw_by_date[as_of][alias] = series
        valid = series.dropna()
        coverage_rows.append({"as_of": as_of, "alias": alias, **series.attrs,
                              "first_observation": valid.index.min(),
                              "last_observation": valid.index.max(),
                              "valid_count": len(valid), "missing_count": int(series.isna().sum()),
                              "last_value": valid.iloc[-1]})
coverage = pd.DataFrame(coverage_rows)
display(coverage[["as_of", "series_id", "first_observation", "last_observation", "valid_count", "missing_count", "last_value"]])
display(coverage[["as_of", "series_id", "vintage_date", "retrieved_at_utc", "sha256", "url"]])

,as_of,series_id,first_observation,last_observation,valid_count,missing_count,last_value
0,2019-08-30,DGS10,2017-01-03,2019-08-29,666,27,1.500
1,2019-08-30,CPIAUCSL,2017-01-01,2019-07-01,31,0,256.161
2,2019-08-30,DFEDTARU,2017-01-01,2019-08-29,971,0,2.250
3,2020-03-31,DGS10,2017-01-03,2020-03-30,810,35,0.700
4,2020-03-31,CPIAUCSL,2017-01-01,2020-02-01,38,0,259.050
5,2020-03-31,DFEDTARU,2017-01-01,2020-03-30,1185,0,0.250
6,2021-12-30,DGS10,2017-01-03,2021-12-29,1249,53,1.550
7,2021-12-30,CPIAUCSL,2017-01-01,2021-11-01,59,0,278.880
8,2021-12-30,DFEDTARU,2017-01-01,2021-12-29,1824,0,0.250
9,2022-10-31,DGS10,2017-01-03,2022-10-28,1458,61,4.020


,as_of,series_id,vintage_date,retrieved_at_utc,sha256,url
0,2019-08-30,DGS10,2019-08-30,2026-09-23T05:24:12.470014+00:00,4f39f04fbf023956b512f4e388d2eb7492f8d30674877f...,https://alfred.stlouisfed.org/graph/alfredgrap...
1,2019-08-30,CPIAUCSL,2019-08-30,2026-09-23T05:24:12.966648+00:00,852d076bcab560baf4e114c8f5eae2155a3fbfaf3c32b3...,https://alfred.stlouisfed.org/graph/alfredgrap...
2,2019-08-30,DFEDTARU,2019-08-30,2026-09-23T05:24:13.585307+00:00,71abe2f4ea6c9b162782c76a7b3b79ecfa858db5143be8...,https://alfred.stlouisfed.org/graph/alfredgrap...
3,2020-03-31,DGS10,2020-03-31,2026-09-23T05:24:14.205939+00:00,21a5b2f08a2555954635e91ff5620f571424b7c53b9873...,https://alfred.stlouisfed.org/graph/alfredgrap...
4,2020-03-31,CPIAUCSL,2020-03-31,2026-09-23T05:24:14.767374+00:00,8de14938f4f81fd0262b2bbcba0c001d32a91120a0301c...,https://alfred.stlouisfed.org/graph/alfredgrap...
5,2020-03-31,DFEDTARU,2020-03-31,2026-09-23T05:24:15.519233+00:00,7b6973087f90db8ee685e26f29c961c198c1575278a388...,https://alfred.stlouisfed.org/graph/alfredgrap...
6,2021-12-30,DGS10,2021-12-30,2026-09-23T05:24:16.236847+00:00,e322d953f49601354ade29c09f6f2f21998142ca07bcb6...,https://alfred.stlouisfed.org/graph/alfredgrap...
7,2021-12-30,CPIAUCSL,2021-12-30,2026-09-23T05:24:16.697445+00:00,f877f659283bab7bccfae6ccd2e2ff22b343f1c9febe64...,https://alfred.stlouisfed.org/graph/alfredgrap...
8,2021-12-30,DFEDTARU,2021-12-30,2026-09-23T05:24:17.436596+00:00,075a2ff5f3090bbd73d74a9e43e64e855f0496495080db...,https://alfred.stlouisfed.org/graph/alfredgrap...
9,2022-10-31,DGS10,2022-10-31,2026-09-23T05:24:18.439185+00:00,e55b5b9bb2ec35aae23012083505b37adef2b4d68697ab...,https://alfred.stlouisfed.org/graph/alfredgrap...


## Feature boundary

Features expose current and base smoothed levels (or CPI YoY rates), their exact window dates, and CPI denominator window dates. Component calculations consume this result, without rereading raw data. All history and freshness checks run for each candidate date.

In [3]:
features_by_date = {as_of: calculate_features(raw, config, as_of)
                    for as_of, raw in raw_by_date.items()}
feature_table = pd.concat(features_by_date, names=["candidate_as_of"])
display(feature_table)

as_of series_id vintage_date  \
candidate_as_of feature                                                        
2019-08-30      yield_trend_endpoints      2019-08-30     DGS10   2019-08-30   
                yield_move_endpoints       2019-08-30     DGS10   2019-08-30   
                inflation_trend_endpoints  2019-08-30  CPIAUCSL   2019-08-30   
                policy_direction_endpoints 2019-08-30  DFEDTARU   2019-08-30   
2020-03-31      yield_trend_endpoints      2020-03-31     DGS10   2020-03-31   
                yield_move_endpoints       2020-03-31     DGS10   2020-03-31   
                inflation_trend_endpoints  2020-03-31  CPIAUCSL   2020-03-31   
                policy_direction_endpoints 2020-03-31  DFEDTARU   2020-03-31   
2021-12-30      yield_trend_endpoints      2021-12-30     DGS10   2021-12-30   
                yield_move_endpoints       2021-12-30     DGS10   2021-12-30   
                inflation_trend_endpoints  2021-12-30  CPIAUCSL   2021-12-30   
                policy_direction_endpoints 2021-12-30  DFEDTARU   2021-12-30   
2022-10-31      yield_trend_endpoints      2022-10-31     DGS10   2022-10-31   
                yield_move_endpoints       2022-10-31     DGS10   2022-10-31   
                inflation_trend_endpoints  2022-10-31  CPIAUCSL   2022-10-31   
                policy_direction_endpoints 2022-10-31  DFEDTARU   2022-10-31   
2023-10-31      yield_trend_endpoints      2023-10-31     DGS10   2023-10-31   
                yield_move_endpoints       2023-10-31     DGS10   2023-10-31   
                inflation_trend_endpoints  2023-10-31  CPIAUCSL   2023-10-31   
                policy_direction_endpoints 2023-10-31  DFEDTARU   2023-10-31   
2024-09-30      yield_trend_endpoints      2024-09-30     DGS10   2024-09-30   
                yield_move_endpoints       2024-09-30     DGS10   2024-09-30   
                inflation_trend_endpoints  2024-09-30  CPIAUCSL   2024-09-30   
                policy_direction_endpoints 2024-09-30  DFEDTARU   2024-09-30   

                                             current      base  \
candidate_as_of feature                                          
2019-08-30      yield_trend_endpoints       1.504000  2.238000   
                yield_move_endpoints        1.504000  2.060000   
                inflation_trend_endpoints   1.814012  2.001152   
                policy_direction_endpoints  2.250000  2.500000   
2020-03-31      yield_trend_endpoints       0.794000  1.906000   
                yield_move_endpoints        0.794000  1.294000   
                inflation_trend_endpoints   2.318104  2.043046   
                policy_direction_endpoints  0.250000  1.750000   
2021-12-30      yield_trend_endpoints       1.496000  1.402000   
                yield_move_endpoints        1.496000  1.588000   
                inflation_trend_endpoints   6.880468  5.202477   
                policy_direction_endpoints  0.250000  0.250000   
2022-10-31      yield_trend_endpoints       4.074000  2.750000   
                yield_move_endpoints        4.074000  3.792000   
                inflation_trend_endpoints   8.222410  8.995221   
                policy_direction_endpoints  3.250000  2.500000   
2023-10-31      yield_trend_endpoints       4.872000  3.942000   
                yield_move_endpoints        4.872000  4.550000   
                inflation_trend_endpoints   3.689903  3.092003   
                policy_direction_endpoints  5.500000  5.500000   
2024-09-30      yield_trend_endpoints       3.764000  4.290000   
                yield_move_endpoints        3.764000  3.832000   
                inflation_trend_endpoints   2.591227  3.250210   
                policy_direction_endpoints  5.000000  5.500000   

                                            lag_observations  smoothing  \
candidate_as_of feature                                                   
2019-08-30      yield_trend_endpoints                     63          5   
                yield_mov

## Component Value / State boundary

Each Component Value is the scaled difference between its Feature endpoints. The separate State is an economic classification of that change. Values, units, thresholds, and Feature references remain visible.

In [4]:
components_by_date = {as_of: calculate_components(features, config)
                      for as_of, features in features_by_date.items()}
component_table = pd.concat(components_by_date, names=["candidate_as_of"])
display(component_table)

as_of       value  \
candidate_as_of component                                           
2019-08-30      Long-End Yield Trend       2019-08-30  -73.400000   
                Recent Long-End Yield Move 2019-08-30  -55.600000   
                Inflation Trend            2019-08-30   -0.187140   
                Policy Direction           2019-08-30  -25.000000   
2020-03-31      Long-End Yield Trend       2020-03-31 -111.200000   
                Recent Long-End Yield Move 2020-03-31  -50.000000   
                Inflation Trend            2020-03-31    0.275059   
                Policy Direction           2020-03-31 -150.000000   
2021-12-30      Long-End Yield Trend       2021-12-30    9.400000   
                Recent Long-End Yield Move 2021-12-30   -9.200000   
                Inflation Trend            2021-12-30    1.677992   
                Policy Direction           2021-12-30    0.000000   
2022-10-31      Long-End Yield Trend       2022-10-31  132.400000   
                Recent Long-End Yield Move 2022-10-31   28.200000   
                Inflation Trend            2022-10-31   -0.772810   
                Policy Direction           2022-10-31   75.000000   
2023-10-31      Long-End Yield Trend       2023-10-31   93.000000   
                Recent Long-End Yield Move 2023-10-31   32.200000   
                Inflation Trend            2023-10-31    0.597899   
                Policy Direction           2023-10-31    0.000000   
2024-09-30      Long-End Yield Trend       2024-09-30  -52.600000   
                Recent Long-End Yield Move 2024-09-30   -6.800000   
                Inflation Trend            2024-09-30   -0.658983   
                Policy Direction           2024-09-30  -50.000000   

                                                        units       state  \
candidate_as_of component                                                   
2019-08-30      Long-End Yield Trend                       bp     falling   
                Recent Long-End Yield Move                 bp     falling   
                Inflation Trend             percentage_points      stable   
                Policy Direction                           bp      easing   
2020-03-31      Long-End Yield Trend                       bp     falling   
                Recent Long-End Yield Move                 bp     falling   
                Inflation Trend             percentage_points     heating   
                Policy Direction                           bp      easing   
2021-12-30      Long-End Yield Trend                       bp      stable   
                Recent Long-End Yield Move                 bp      stable   
                Inflation Trend             percentage_points     heating   
                Policy Direction                           bp   unchanged   
2022-10-31      Long-End Yield Trend                       bp      rising   
                Recent Long-End Yield Move                 bp      rising   
                Inflation Trend             percentage_points     cooling   
                Policy Direction                           bp  tightening   
2023-10-31      Long-End Yield Trend                       bp      rising   
                Recent Long-End Yield Move                 bp      rising   
                Inflation Trend             percentage_points     heating   
                Policy Direction                           bp   unchanged   
2024-09-30      Long-End Yield Trend                       bp     falling   
                Recent Long-End Yield Move                 bp      stable   
                Inflation Trend             percentage_points     cooling   
                Policy Direction                           bp      easing   

                                            threshold  \
candidate_as_of component                               
2019-08-30      Long-End Yield Trend            25.00   
                Recent Long-End Yield Move      15.00   
                Inflat

## Duration Bond Exposure View boundary and historical comparison

Only Component results enter View calculation. Each table entry shows `Value / State`; the View retains all individual contributions and its Rule Case.

In [5]:
views_by_date = {as_of: calculate_view(components, config)
                 for as_of, components in components_by_date.items()}
comparison_rows = []
for as_of, components in components_by_date.items():
    row = {"as_of": as_of}
    for name, component in components.iterrows():
        row[name] = f"{component['value']:.2f} {component['units']} / {component['state']}"
    row.update({key: views_by_date[as_of][key] for key in ("tally", "view")})
    comparison_rows.append(row)
comparison = pd.DataFrame(comparison_rows).set_index("as_of")
display(comparison)
display(pd.DataFrame({date: view["contributions"] for date, view in views_by_date.items()}).T)

,Long-End Yield Trend,Recent Long-End Yield Move,Inflation Trend,Policy Direction,tally,view
as_of,,,,,,
2019-08-30,-73.40 bp / falling,-55.60 bp / falling,-0.19 percentage_points / stable,-25.00 bp / easing,3,duration_supportive
2020-03-31,-111.20 bp / falling,-50.00 bp / falling,0.28 percentage_points / heating,-150.00 bp / easing,2,duration_supportive
2021-12-30,9.40 bp / stable,-9.20 bp / stable,1.68 percentage_points / heating,0.00 bp / unchanged,-1,mixed
2022-10-31,132.40 bp / rising,28.20 bp / rising,-0.77 percentage_points / cooling,75.00 bp / tightening,-2,duration_headwind
2023-10-31,93.00 bp / rising,32.20 bp / rising,0.60 percentage_points / heating,0.00 bp / unchanged,-3,duration_headwind
2024-09-30,-52.60 bp / falling,-6.80 bp / stable,-0.66 percentage_points / cooling,-50.00 bp / easing,3,duration_supportive


,Long-End Yield Trend,Recent Long-End Yield Move,Inflation Trend,Policy Direction
2019-08-30,1,1,0,1
2020-03-31,1,1,-1,1
2021-12-30,0,0,-1,0
2022-10-31,-1,-1,1,-1
2023-10-31,-1,-1,-1,0
2024-09-30,1,0,1,1


## Backward lineage inspection

Change `inspect_as_of` and `inspect_component` below. The View references Component States; the chosen Component references its Feature; the Feature identifies the exact Raw Observation windows. CPI windows include the year-earlier denominator observations. Full inputs and outputs for every date remain in the dictionaries above.

In [6]:
inspect_as_of = "2021-12-30"
inspect_component = "Inflation Trend"
display(views_by_date[inspect_as_of])
component = components_by_date[inspect_as_of].loc[inspect_component]
display(component.to_frame("component_result"))
feature = features_by_date[inspect_as_of].loc[component["feature"]]
display(feature.to_frame("feature_result"))
feature_spec = config["features"][component["feature"]]
raw = raw_by_date[inspect_as_of][feature_spec["series"]]
windows = {}
for endpoint in ("current", "base"):
    windows[endpoint] = raw.loc[feature[f"{endpoint}_start"]:feature[f"{endpoint}_end"]]
    if pd.notna(feature[f"{endpoint}_denominator_start"]):
        windows[endpoint + "_denominator"] = raw.loc[
            feature[f"{endpoint}_denominator_start"]:feature[f"{endpoint}_denominator_end"]]
display(pd.concat(windows, names=["window", "observation_date"]).to_frame("raw_value"))
display(raw.attrs)

{'as_of': Timestamp('2021-12-30 00:00:00'),
 'view': 'mixed',
 'tally': -1,
 'rule_case': {'Long-End Yield Trend': 'stable',
  'Recent Long-End Yield Move': 'stable',
  'Inflation Trend': 'heating',
  'Policy Direction': 'unchanged'},
 'contributions': {'Long-End Yield Trend': 0,
  'Recent Long-End Yield Move': 0,
  'Inflation Trend': -1,
  'Policy Direction': 0}}

,component_result
as_of,2021-12-30 00:00:00
value,1.677992
units,percentage_points
state,heating
threshold,0.25
feature,inflation_trend_endpoints


,feature_result
as_of,2021-12-30 00:00:00
series_id,CPIAUCSL
vintage_date,2021-12-30 00:00:00
current,6.880468
base,5.202477
lag_observations,3
smoothing,1
transform,year_over_year
current_start,2021-11-01 00:00:00
current_end,2021-11-01 00:00:00


,,raw_value
window,observation_date,
current,2021-11-01,278.880
current_denominator,2020-11-01,260.927
base,2021-08-01,273.012
base_denominator,2020-08-01,259.511


{'series_id': 'CPIAUCSL',
 'vintage_date': '2021-12-30',
 'url': 'https://alfred.stlouisfed.org/graph/alfredgraph.csv?id=CPIAUCSL&cosd=2017-01-01&coed=2021-12-30&vintage_date=2021-12-30',
 'source': 'Bureau of Labor Statistics, seasonally adjusted headline CPI',
 'units': 'index_1982_84_100',
 'retrieved_at_utc': '2026-09-23T05:24:16.697445+00:00',
 'sha256': 'f877f659283bab7bccfae6ccd2e2ff22b343f1c9febe641a00658d99feb67d1b'}

## Determinism and diagnostic sensitivity

Recalculate every boundary and compare exact results. Then independently vary Component thresholds ±20% using the same Features. These sensitivity results are diagnostic only and do not feed the base Views. They do not establish predictive usefulness or validate other horizons / proxies.

In [7]:
from copy import deepcopy

for as_of, raw in raw_by_date.items():
    features = calculate_features(raw, config, as_of)
    components = calculate_components(features, config)
    pd.testing.assert_frame_equal(features, features_by_date[as_of])
    pd.testing.assert_frame_equal(components, components_by_date[as_of])
    assert calculate_view(components, config) == views_by_date[as_of]
assert list(views_by_date) == config["as_of_dates"]
assert all(components["value"].notna().all() for components in components_by_date.values())
print("PASS: coverage, finite Features / Components, all six Views, deterministic recalculation")

sensitivity_rows = []
for factor in (0.8, 1.0, 1.2):
    diagnostic_config = deepcopy(config)
    for spec in diagnostic_config["components"].values():
        spec["threshold"] *= factor
    for as_of, features in features_by_date.items():
        result = calculate_view(calculate_components(features, diagnostic_config), diagnostic_config)
        sensitivity_rows.append({"as_of": as_of, "threshold_factor": factor, "view": result["view"]})
sensitivity = pd.DataFrame(sensitivity_rows).pivot(index="as_of", columns="threshold_factor", values="view")
display(sensitivity)

PASS: coverage, finite Features / Components, all six Views, deterministic recalculation


threshold_factor,0.8,1.0,1.2
as_of,,,
2019-08-30,duration_supportive,duration_supportive,duration_supportive
2020-03-31,duration_supportive,duration_supportive,duration_supportive
2021-12-30,mixed,mixed,mixed
2022-10-31,duration_headwind,duration_headwind,duration_headwind
2023-10-31,duration_headwind,duration_headwind,duration_headwind
2024-09-30,duration_supportive,duration_supportive,duration_supportive


## Review before Stage 2

Review the historical table and lineage, rather than declaring success from execution alone. The 2021 policy-transition case can remain `unchanged` because this proxy measures realized policy. High yield levels and high inflation levels are not represented by these change Components, so 2023 restrictiveness need not itself produce a headwind. Shock-driven falling yields are treated as supportive conditions without a valuation or liquidity assessment. Threshold sensitivity and overlapping yield signals merit economic review; distinct output labels alone do not establish useful interpretation beyond yield momentum.

Stage 2 is deferred. Any future ETF-specific Duration Evaluation must consume authoritative Components directly, not these Views.


Executed results: 2019/2020/2024 are supportive, 2021 is mixed, and 2022/2023 are headwinds. All six Views are unchanged under ±20% Component threshold sensitivity. The 2020 inflation signal uses February CPI (the latest available reference month). See [the implementation review](reports/260923_duration_stage1_review.md) for the full result table, validation record, and provisional assumptions.